In [1]:
//#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadTestingIO\BoSSSpad.dll"
//#r "C:\Users\miao\Documents\JupyterNotebook\BoSSSpadTestingIOAfter\BoSSSpad.dll"
#r "D:\Users\miao\Documents\JupyterNotebook\BoSSSpadMiao\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using System.IO;
using System.Data;
using System.Text;
using System.Globalization;
using System.Threading;
using System.Threading.Tasks;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
using BoSSS.Foundation.Grid.RefElements;
using BoSSS.Platform.LinAlg;
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using ZwoLevelSetSolver;
using ZwoLevelSetSolver.SolidPhase;
Init();

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
//ExecutionQueues

In [3]:
//GetDefaultQueue()

In [4]:
static var myBatch = ExecutionQueues[0];
static var myDb = myBatch.CreateOrOpenCompatibleDatabase("EE-FS-Couette-Flow");

In [5]:
BoSSSshell.WorkflowMgm.Init("EE-FS-Couette-Flow");

Project name is set to 'EE-FS-Couette-Flow'.
Default Execution queue is chosen for the database.
Opening existing database '\\dc3\userspace\miao\cluster\EE-FS-Couette-Flow'.


In [6]:
BoSSSshell.WorkflowMgm.DefaultDatabase = myDb;

Opening existing database 'D:\Users\miao\Documents\Database\EE-FS-Couette-Flow'.


## Create grid

In [24]:
public static class GridFactory {
 
    public static Grid2D GenerateGrid(int h) { 
        double xLeft = -1;
        double xRight = 1;
        double ySize = 2;
        int kelem = Convert.ToInt32(Math.Pow(2, h));;

        double[] Xnodes = GenericBlas.Linspace(xLeft, xRight, kelem + 1);
        double[] Ynodes = GenericBlas.Linspace(0, ySize, kelem + 1);
        var grd = Grid2D.Cartesian2DGrid(Xnodes, Ynodes, 
                                        periodicX: true, periodicY: false);

        grd.EdgeTagNames.Add(1, "wall_lower");
        grd.EdgeTagNames.Add(2, "wall_upper");
        //grd.EdgeTagNames.Add(3, "velocity_inlet_left");
        //grd.EdgeTagNames.Add(4, "pressure_outlet_right");

        grd.DefineEdgeTags(delegate (double[] X) {
            byte et = 0;

            if(Math.Abs(X[1] - 0) <= 1.0e-8)
                et = 1;
            if(Math.Abs(X[1] - ySize) <= 1.0e-8)
                et = 2;
//            if(Math.Abs(X[0] - xLeft) <= 1.0e-8)
//                et = 3;
//            if(Math.Abs(X[0] - xRight) <= 1.0e-8)
//                et = 4;

            return et;
        });

        return grd;
     }
 
 }

In [26]:
public static class BoundaryValueFactory { 
    public static string GetPrefixCode() {
        using(var stw = new System.IO.StringWriter()) {
           
           stw.WriteLine("static class BoundaryValues {");
           stw.WriteLine("  static public double Zero(double[] X) {");
           stw.WriteLine("    return 0.0;");
           stw.WriteLine("  }");

           stw.WriteLine("  static public double InletVelocity(double[] X, double t) {");
           stw.WriteLine("    return 1.0;");
           stw.WriteLine("  }");
           
           stw.WriteLine(" static public double InitialPhi(double[] X) { ");
           stw.WriteLine("    return -1;");
           stw.WriteLine("    }"); 
           
           stw.WriteLine(" static public double InitialPhi1(double[] X) { ");
           stw.WriteLine("    return -(X[1] - 1);");
           stw.WriteLine("    }"); 

           stw.WriteLine("}"); 
           return stw.ToString();
        }
    }
   
    static public Formula Get_Zero() {
        return new Formula("BoundaryValues.Zero", AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_InletVelocity(){
        return new Formula("BoundaryValues.InletVelocity", true, AdditionalPrefixCode:GetPrefixCode());
    }

    static public Formula Get_InitialPhi(){
        return new Formula("BoundaryValues.InitialPhi", AdditionalPrefixCode:GetPrefixCode());
    }
    
    static public Formula Get_InitialPhi1(){
        return new Formula("BoundaryValues.InitialPhi1", AdditionalPrefixCode:GetPrefixCode());
    }
}

## Create Control file

In [29]:
public static ZLS_Control Couette( int p = 2, int h = 2, int AMRlvl = 2, double BeamDensity = 1) {
    ZLS_Control C = new ZLS_Control(p);
    C.AgglomerationThreshold = 0.2;
    C.NoOfMultigridLevels = 1;
    //int D = 2;
    AppControl._TimesteppingMode compMode = AppControl._TimesteppingMode.Transient;

    #region db
    C.savetodb = true;
    C.ProjectName = "Couette";
    C.SessionName = "Fluid-Solid-Couette-p" + p + "-h" + h;
    C.ContinueOnIoError = false;
    //C.LogValues = XNSE_Control.LoggingValues.MovingContactLine;
    //C.PostprocessingModules.Add(new MovingContactLineLogging());
    #endregion
    
    // Physical Parameters
    // ===================
    #region physics
    C.PhysicalParameters.rho_A = 1.0;
    //C.PhysicalParameters.rho_B = 1.0;
    C.PhysicalParameters.mu_A = 10;
    //C.PhysicalParameters.mu_B = 10;
    //double sigma = 0.046;
    //C.PhysicalParameters.Sigma = sigma;
    //C.PhysicalParameters.betaS_A = 0.0;
    //C.PhysicalParameters.betaS_B = 0.0;
    //C.PhysicalParameters.betaL = 0.0;
    //C.PhysicalParameters.theta_e = Math.PI / 2.0;
    C.PhysicalParameters.IncludeConvection = true;
    C.PhysicalParameters.Material = false; //??
    C.Material = new Solid() {
        Density = BeamDensity,
        //Lame2 = 1000,
        //Viscosity = 0.1
        Lame2 = 1e1,
        Viscosity = 0.05
    };
    #endregion
    // grid generation
    // ===============
    #region grid
    C.SetGrid(GridFactory.GenerateGrid(h));
    #endregion
    // Initial Values
    // ==============
    //
    C.AddInitialValue("VelocityX#A", BoundaryValueFactory.Get_Zero());
    C.AddInitialValue("VelocityX#B", BoundaryValueFactory.Get_Zero());
    C.AddInitialValue("Phi", BoundaryValueFactory.Get_InitialPhi());
    C.AddInitialValue("Phi2", BoundaryValueFactory.Get_InitialPhi1());
    //C.AddInitialValue(ZwoLevelSetSolver.VariableNames.SolidLevelSetCG, BoundaryValueFactory.Get_InitialPhi1());
    // boundary conditions
    // ===================
    #region BC
    C.AddBoundaryValue("wall_lower");
    C.AddBoundaryValue("wall_upper", "VelocityX#A", BoundaryValueFactory.Get_InletVelocity());
    C.AddBoundaryValue("wall_upper", "VelocityX#B", BoundaryValueFactory.Get_InletVelocity());
    //C.AddBoundaryValue("pressure_outlet_right");
    C.AdvancedDiscretizationOptions.GNBC_Localization = NavierSlip_Localization.Bulk;
    C.AdvancedDiscretizationOptions.GNBC_SlipLength = NavierSlip_SlipLength.Prescribed_Beta;
    //C.PhysicalParameters.sliplength = 0.001;
    #endregion
    // misc. solver options
    // ====================
    #region solver
    //C.AdvancedDiscretizationOptions.CellAgglomerationThreshold = 0.2;
    //C.AdvancedDiscretizationOptions.PenaltySafety = 40;
    //C.AdvancedDiscretizationOptions.UseGhostPenalties = true;
    C.NonLinearSolver.MaxSolverIterations = 80;
    C.NonLinearSolver.MinSolverIterations = 2;
    //C.Solver_MaxIterations = 50;
    C.NonLinearSolver.ConvergenceCriterion = 1e-8;
    //C.Solver_ConvergenceCriterion = 1e-8;
    C.LevelSet_ConvergenceCriterion = 1e-12;
    //C.NonLinearSolver.Globalization = BoSSS.Solution.AdvancedSolvers.Newton.GlobalizationOption.Dogleg;
    //C.Option_LevelSetEvolution = (compMode == AppControl._TimesteppingMode.Steady) ? LevelSetEvolution.None : LevelSetEvolution.FastMarching;
    //C.EllipticExtVelAlgoControl.solverFactory = () => new ilPSP.LinSolvers.PARDISO.PARDISOSolver();
    //C.EllipticExtVelAlgoControl.IsotropicViscosity = 1e-3;
    //C.fullReInit = false; 
    C.AdvancedDiscretizationOptions.FilterConfiguration = CurvatureAlgorithms.FilterConfiguration.NoFilter;
    C.AdvancedDiscretizationOptions.ViscosityMode = ViscosityMode.Standard;
    C.AdaptiveMeshRefinement = false;
    C.activeAMRlevelIndicators.Add(new AMRonNarrowband { maxRefinementLevel = AMRlvl });
    C.AMR_startUpSweeps = AMRlvl;
    #endregion

    C.DynamicLoadBalancing_On = false;
    //C.DynamicLoadBalancing_Period = 500;
    C.DynamicLoadBalancing_RedistributeAtStartup = false;

    //C.GridPartType = GridPartType.clusterHilbert;
    C.GridPartType = GridPartType.METIS;

    // Timestepping
    // ============
    #region time
    //C.CheckJumpConditions = true;
    C.TimeSteppingScheme = TimeSteppingScheme.BDF2;
    C.Timestepper_BDFinit = TimeStepperInit.SingleInit;
    C.Timestepper_LevelSetHandling = LevelSetHandling.LieSplitting;
    C.NonLinearSolver.SolverCode = NonLinearSolverCode.Newton;
    
    C.TimesteppingMode = compMode;
    double dt = 1e0;
    C.dtMax = dt;
    C.dtMin = dt;
    C.NoOfTimesteps = 50;
    C.saveperiod = 1;
    #endregion
    return C;
}

## Send and run jobs

In [32]:
double[] Density = new double[] {1}; 
int[] Resolution = new int[] {3}; 
int[] DgOrder = new int[] {2}; 

In [34]:
foreach(int DgO in DgOrder){
foreach(int Res in Resolution){
foreach(double Den in Density){
    var C_CTRLFILE = Couette(DgO, Res, 0, Den);//Create control file.
    var JobLocal = C_CTRLFILE.CreateJob();
    JobLocal.NumberOfMPIProcs = 1;
    JobLocal.NumberOfThreads = 1;
    JobLocal.Activate(myBatch);
    JobLocal.ShowOutput(); 
}
}
}

Grid Edge Tags changed.
Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job Fluid-Solid-Couette-p2-h3 ... 
Set Database: { Session Count = 2; Grid Count = 2; Path = D:\Users\miao\Documents\Database\EE-FS-Couette-Flow }
Grid is not in database yet...
Grid successfully saved: 62a49e69-5b77-479e-8c99-5d04bfd66f7d


Deploying executables and additional files ...
Deployment directory: D:\Users\miao\Documents\Database\Deployment\EE-FS-Couette-Flow-ZwoLevelSetSolver2024Aug23_100540.941197
copied 46 files.
   written file: control.obj
deployment finished.
Mini batch processor is already running.
Starting external console ...
(You may close the new window at any time, the job will continue.)


In [13]:
wmg.Sessions

#0: EE-FS-Couette-Flow	Fluid-Solid-Couette-p2-h2	08/23/2024 10:00:17	94da2e65...
#1: EE-FS-Couette-Flow	Fluid-Solid-Couette-p2-h3	08/23/2024 09:15:44	c1b62599...


In [15]:
wmg.Sessions[0].Timesteps.Count

51

In [36]:
var outPath = wmg.Sessions[0].Timesteps.Every(1).Skip(0).Export().WithSupersampling(3).Do();

Starting export process... Data will be written to the directory: d:\Users\miao\AppData\Local\BoSSS\plots\sessions\EE-FS-Couette-Flow__Fluid-Solid-Couette-p2-h3__50ce5c57-3d9d-43dd-86ed-12f41fc2b761


## Post processing

In [ ]:
//var f = databases.Pick(0).Sessions.Pick(0).Timesteps.Pick(1).GetField("Phi");

In [ ]:
//var v = databases.Pick(0).Sessions.Pick(0).Timesteps.Pick(5).GetField("VelocityX");

In [59]:
wmg.Sessions

#0: EE-FS-Couette-Flow	Fluid-Solid-Couette-p2-h2*	08/23/2024 10:00:17	94da2e65...
#1: EE-FS-Couette-Flow	Fluid-Solid-Couette-p2-h3	08/23/2024 09:15:44	c1b62599...


In [61]:
var session = wmg.Sessions[0];

In [63]:
session.Timesteps.Count

50

In [65]:
var mydataset = session.Timesteps.ToDataSet(
        t => t.PhysicalTime,
        t => t.Fields.Find("DisplacementX").ProbeAt(0.0, 0.25),
        t => "DisplacementX"
        );

In [67]:
mydataset.PlotNow()

Using gnuplot: d:\Users\miao\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 0.05 
 
 
 
 
 0.1 
 
 
 
 
 0.15 
 
 
 
 
 0.2 
 
 
 
 
 0.25 
 
 
 
 
 0.3 
 
 
 
 
 0 
 
 
 
 
 10 
 
 
 
 
 20 
 
 
 
 
 30 
 
 
 
 
 40 
 
 
 
 
 50 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 DisplacementX 
 
 
 DisplacementX 
 
 
 
	<path stroke='rgb( 0, 0, 0)' d='M533.0,57.1 L586.4,57.1 M62.2,564.0 L76.0,358.3 L89.7,209.5 L103.5,145.5 L117.2,124.9 L131.0,120.3
 L144.7,120.9 L158.5,122.4 L172.3,123.4 L186.0,123.9 L199.8,124.1 L213.5,124.1 L227.3,124.1 L241.1,124.1
 L254.8,124.1 L268.6,124.1 L282.3,124.1 L296.1,124.1 L309.8,124.1 L323.6,124.1 L337.4,124.1 L351.1,124.1
 L364.9,124.1 L378.6,124.1 L392.4,124.1 L406.2,124.1 L419.9,124.1 L433.7,124.1 L447.4,124.1 L461.2,124.1
 L474.9,124.1 L488.7,124.1 L502.5,124.1 L516.2,124.1 L530.0,124.1 L543.7,124.1 L557.5,124.1 L571.2,124.1
 L585.0,124.1 L598.8,124.1 L612.5,124.1 L626.3,124.1 L640.0,124.1 L653.8,124.1 L667.6,124.1 L681.3,124.1
 L695.1,124.1 L708.8,124.1 L722.6,124.1 L736.3,124.1 L750.1,124.1 '/>

In [149]:
for (int i = 0; i < 5; i++) 
{
  var DispX = wmg.Sessions[i].Timesteps.Last().Fields.Where(f=>f.Identification == "DisplacementX").Single();
  var VeloX = wmg.Sessions[i].Timesteps.Last().Fields.Where(f=>f.Identification == "VelocityX").Single();
  Console.WriteLine(DispX.ProbeAt(0.0, 0.25));
  //Console.WriteLine(VeloX.ProbeAt(0.0, 1.0));
}

0.0008327539119221423
0.000832753905586246
0.0008327538562921613
0.0008327534738945147
0.0008327505542550956


In [132]:
int[] Cases = new int[] {0, 1, 3, 4}; 

In [133]:
foreach(int i in Cases){
  var DispX = wmg.Sessions[i].Timesteps.Last().Fields.Where(f=>f.Identification == "DisplacementX").Single();
  var VeloX = wmg.Sessions[i].Timesteps.Last().Fields.Where(f=>f.Identification == "VelocityX").Single();
  Console.WriteLine(DispX.ProbeAt(0.0, 0.25));
  Console.WriteLine(VeloX.ProbeAt(0.0, 1.0));
}

0.000832753905586246
0.3337694306042147
0.0008327538562921613
0.3337694745333356
0.0008327534738945147
0.33376982641370684
0.0008327505542550956
0.3337725817949061


In [50]:
var DispX = session.Timesteps.Last().Fields.Where(f=>f.Identification == "DisplacementX").Single();

In [51]:
DispX.ProbeAt(0.0, 0.25)

0.0008327781162742156

In [53]:
Displacement


(1,1): error CS0103: The name 'Displacement' does not exist in the current context



Error: compilation error

## Restart

In [ ]:
databases[0].Sessions

In [ ]:
//var Sloc = databases[0].Sessions.Last();
var Sloc = databases[0].Sessions[0];
Sloc

In [ ]:
var c2 = Sloc.CreateRestartControl() as ZLS_Control;

In [ ]:
c2.GetType()

In [ ]:
c2.GridGuid

In [ ]:
Sloc.Timesteps.Last().GridID

In [ ]:
c2.GridGuid = Sloc.Timesteps.Last().GridID;

In [ ]:
c2.GridGuid

In [ ]:
//c2.DynamicLoadBalancing_On = true;
//c2.DynamicLoadBalancing_Period = 10;
c2.DynamicLoadBalancing_RedistributeAtStartup = false;
//c2.GridPartType = GridPartType.METIS;

//c2.AdaptiveMeshRefinement = true;
//c2.activeAMRlevelIndicators.Add(new ContactPointRefiner { maxRefinementLevel = 5});
//c2.AMR_startUpSweeps = 5;
c2.AdaptiveMeshRefinement = false;

In [ ]:
//c2.TimeSteppingScheme = TimeSteppingScheme.BDF2;

In [ ]:
var JobLocal2 = c2.CreateJob();
JobLocal2.Status

In [ ]:
JobLocal2.NumberOfMPIProcs = 2;
JobLocal2.Activate();
JobLocal2.ShowOutput();

In [ ]:
databases[0].Sessions

In [ ]:
var outPath = databases[0].Sessions[0].Timesteps.Every(1).Skip(0).Export().WithSupersampling(3).Do();

In [ ]:
databases[0].Sessions[0].